[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day3_live.ipynb)

# Day 3 · 강의 — 머신러닝

경사 하강법 · scikit-learn · 검증과 평가 · 회귀

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

모든 셀에 **코드가 채워져 있다.** 위에서부터 실행해 결과를 눈으로 확인한다.
강사가 설명하는 동안 값을 바꿔 가며 돌려 본다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 학습의 원리

> **실습문제 1.** 학습률 `lr` 을 `0.01` 로 두고 20회를 돌린 뒤 `x` 를 확인한다.
값이 3에 **덜 가까워지는 것**을 본다.

In [ ]:
def grad(x): return 2 * (x - 3)
x = 10.0
lr = 0.01
for _ in range(20):
    x = x - lr * grad(x)

assert x > 4, f'학습률이 작으면 20회로는 못 간다. 실제 {x}'
print('통과 — x =', round(x, 3))

> **실습문제 2.** 이번엔 학습률 `lr` 을 `1.1` 로 두고 같은 20회를 돌린다. `x` 가 **발산**한다.

In [ ]:
def grad(x): return 2 * (x - 3)
x = 10.0
lr = 1.1
for _ in range(20):
    x = x - lr * grad(x)

assert abs(x) > 100, f'학습률이 너무 크면 튕겨 나간다. 실제 {x}'
print('통과 — x =', round(x, 1))

## 2. scikit-learn — 네 줄로 끝나는 학습

> **실습문제 3.** 로지스틱 회귀를 학습하고 **테스트 정확도**를 `acc` 에 담는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.linear_model import LogisticRegression
X_tr, X_te, y_tr, y_te = split()
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
model = LogisticRegression(max_iter=1000)
model.fit(X_tr_s, y_tr)
acc = model.score(X_te_s, y_te)

assert acc > 0.85, f'0.85 는 넘어야 한다. 실제 {acc}'
print('통과 — 정확도', round(acc, 3))

## 3. 검증과 평가

> **실습문제 4.** **전부 양품이라 찍는** 예측을 만들어 정확도를 `dumb_acc` 에 담는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
import numpy as np
X_tr, X_te, y_tr, y_te = split()
pred = np.ones(len(y_te), dtype=int)
dumb_acc = (pred == y_te).mean()

assert abs(dumb_acc - 0.81) < 0.02, f'실제 {dumb_acc}'
print('통과 — 아무것도 안 배워도', round(dumb_acc, 3))